# Fine Tuning The Model

In [ ]:
"""
LoRA fine-tuning of a causal LM from Hugging Face Hub on a HF Dataset.
Requires: transformers, datasets, accelerate, peft, bitsandbytes, torch

pip install transformers datasets accelerate peft bitsandbytes torch
"""

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# -----------------------
# Config — edit these
# -----------------------
MODEL_NAME = "BioMistral/BioMistral-7B"           # any causal LM repo id on HF Hub
DATASET_NAME = "lavita/medical-qa-datasets"   # HF dataset repo id
TEXT_FIELD = "text"                      # column in the dataset with training text
OUTPUT_DIR = "./finetuned-model"
MAX_LENGTH = 1024

# -----------------------
# 4-bit quantization config (QLoRA) — keeps a 7B model comfortably under 30GB
# -----------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# -----------------------
# Load tokenizer & model
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

# -----------------------
# LoRA config
# -----------------------
lora_config = LoraConfig(
    r=16,                       # rank — bump to 32/64 for more capacity, more VRAM
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[            # standard for Llama/Qwen/Mistral-family models
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # sanity check — should be << 1% of total params

# -----------------------
# Load & tokenize dataset
# -----------------------
dataset = load_dataset(DATASET_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_FIELD],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

if "validation" in tokenized:
    train_ds, eval_ds = tokenized["train"], tokenized["validation"]
else:
    split = tokenized["train"].train_test_split(test_size=0.05, seed=42)
    train_ds, eval_ds = split["train"], split["test"]

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# -----------------------
# Training arguments
# -----------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,      # effective batch size = 16
    learning_rate=2e-4,                 # LoRA typically wants a higher LR than full FT
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",           # memory-efficient optimizer, pairs well with QLoRA
)

# -----------------------
# Train
# -----------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

trainer.train()

# -----------------------
# Save LoRA adapter (small — just the adapter weights, not the full model)
# -----------------------
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"LoRA adapter saved to {OUTPUT_DIR}")